# Neuro-Symbolic Intrusion Detection for Resource-Constrained IIoT/ICS/SCADA Environments

**A hybrid neural-symbolic framework designed for real-world edge deployment in Industrial Internet of Things (IIoT), Industrial Control Systems (ICS), and SCADA networks.**

### Why Neuro-Symbolic?
Modern IIoT/ICS environments face severe constraints:
- Extremely limited compute and memory on PLCs, RTUs, and edge gateways
- Strict real-time requirements (sub-millisecond decisions)
- High class imbalance and evolving attack patterns
- Need for interpretability and trust in safety-critical systems

Pure deep learning models often achieve high accuracy but lack explainability and are too heavy for edge devices. Pure symbolic/rule-based systems are lightweight and interpretable but struggle with complex temporal patterns.

This project introduces a neuro-symbolic fusion architecture that combines the temporal modelling power of a GRU neural network with a lightweight, rule-calibrated symbolic component, delivering both high detection performance and edge-friendly characteristics.

### Architecture Overview
- **Neural Component**: GRU-based temporal model (sequence-aware)  
- **Symbolic Component**: Shallow, interpretable decision tree / ontology learned from training data  
- **Fusion**: Learned weighted combination of neural and symbolic scores  
- **Target Deployment**: Resource-constrained edge devices (sub-millisecond latency, low memory footprint)

### Datasets Used
- **CIC-DDoS2019** - Volumetric DDoS attacks (SYN, UDP, NTP, TFTP floods)
- **Edge-IIoTset** - Realistic IoT/IIoT attack scenarios
- **CICIoT2023** - Large-scale modern IoT traffic with diverse attacks

### Project Goals
- Achieve **≥ 98% F1-score** across all datasets with the hybrid model
- Maintain **sub-millisecond single-packet inference latency**
- Keep memory footprint low enough for real edge gateway deployment
- Provide **interpretable symbolic rules** alongside high neural performance


**Notebook Structure**
1. Data Loading (CIC-DDoS2019, Edge-IIoTset, CICIoT2023)
2. Feature Engineering & Scaling
3. GRU Training (temporal neural model)
4. Symbolic Ontology Construction
5. Neuro-Symbolic Fusion & Evaluation
6. Publication Figures (bar charts, confusion matrices, ROC/PR curves)
7. Inference Benchmark (latency & throughput)
8. LaTeX-ready tables

*Last updated: May 2026*

In [4]:
# CELL 1: Environment Setup & Global Configuration
import os
import platform
import random
import warnings
import time
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, roc_curve,
    precision_recall_curve
)
from sklearn.cluster import MiniBatchKMeans

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pathlib import Path
from tqdm import tqdm

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("CELL 1: Environment Setup & Global Configuration\n")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device & Hardware Settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CORES = os.cpu_count() or 4

torch.set_num_threads(N_CORES)

# Safe interop threads setting (prevents RuntimeError)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass   # already set by background processes

print(f"Device          : {device}")
print(f"CPU Threads     : {torch.get_num_threads()} (compute) | {torch.get_num_interop_threads()} (interop)")

# Global Hyperparameters
SEQ_LEN        = 16   # Sliding window length
BATCH_TRAIN    = 256
BATCH_INFER    = 512
EPOCHS         = 20
LR             = 3e-3
HIDDEN_SIZE    = 64
GRU_LAYERS     = 2
DROPOUT        = 0.25
N_WORKERS      = min(4, N_CORES)

# Output Directory
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

print(f"Output directory: {OUT.resolve()}")
print(f"SEQ_LEN={SEQ_LEN} | BATCH={BATCH_TRAIN} | EPOCHS={EPOCHS} | HIDDEN={HIDDEN_SIZE}")
print("Setup completed successfully.\n")

CELL 1: Environment Setup & Global Configuration

Device          : cpu
CPU Threads     : 16 (compute) | 1 (interop)
Output directory: F:\jupyter\outputs
SEQ_LEN=16 | BATCH=256 | EPOCHS=20 | HIDDEN=64
Setup completed successfully.



In [5]:
# CELL 2: Dataset Paths
# stratified subsampling is used only when memory would be exceeded.

BASE_PATH = Path(r'F:\jupyter\kagglehub')

DATASET_PATHS = {
    'cic_ddos19': {
        'root': BASE_PATH / r'datasets\dhoogla\cicddos2019\versions\3',
        'split': False
    },
    'edge_iiotset': {
        'root': BASE_PATH / r'edgeiiotset-cyber-security-dataset-of-iot-iiot\versions\5'
                           r'\Edge-IIoTset dataset\Selected dataset for ML and DL',
        'split': False
    },
    'ciciot23': {
        'train': BASE_PATH / r'CICIOT23\train\train.csv',
        'val':   BASE_PATH / r'CICIOT23\validation\validation.csv',
        'test':  BASE_PATH / r'CICIOT23\test\test.csv',
        'split': True
    }
}

print('Dataset paths configured.')
for k, v in DATASET_PATHS.items():
    if v['split']:
        exists = all(Path(v[s]).exists() for s in ['train', 'val', 'test'])
    else:
        exists = v['root'].exists()
    status = 'FOUND' if exists else ' NOT FOUND'
    print(f'  {k}: {status}')

Dataset paths configured.
  cic_ddos19: FOUND
  edge_iiotset: FOUND
  ciciot23: FOUND


In [6]:
# CELL 3: CIC-DDoS / Edge-IIoT / CICIOT23 Loader 
print("CELL 3: Loading datasets \n")

from pathlib import Path
import pandas as pd
import numpy as np

base_path = Path(r"F:\jupyter\kagglehub")
paths = {
    "cic_ddos": base_path / r"datasets\dhoogla\cicddos2019\versions\3",
    "edge_iot": base_path / r"edgeiiotset-cyber-security-dataset-of-iot-iiot\versions\5\Edge-IIoTset dataset\Selected dataset for ML and DL",
    "ciciot23": base_path / r"CICIOT23"
}

datasets = {}

# Label conversion
def to_binary(x):
    if pd.isna(x):
        return 0
    x = str(x).lower()
    return 0 if ("normal" in x or "benign" in x or x == "0") else 1

# File reader
def read_any(f):
    try:
        if f.suffix == ".csv":
            return pd.read_csv(f, low_memory=False)
        elif f.suffix == ".parquet":
            return pd.read_parquet(f)
        else:
            return None
    except Exception as e:
        print(f" [!] Could not read {f.name}: {e}")
        return None

# Numeric column extractor (protects 'target')
def get_numeric_cols(df):
    df = df.copy()
    
    # Saving target column before dropping anything
    target_exists = "target" in df.columns
    target_col = df["target"].copy() if target_exists else None
    
    # Identifying columns to drop (object / category, except target)
    drop_dtypes = ["object", "category"]
    bad_cols = [
        c for c in df.columns
        if c != "target" and (
            df[c].dtype.name in drop_dtypes or 
            hasattr(df[c], "cat") or 
            pd.api.types.is_object_dtype(df[c])
        )
    ]
    
    df.drop(columns=bad_cols, inplace=True)
    
    # Coercing remaining columns to numeric (skipping target)
    for col in list(df.columns):
        if col == "target":
            continue
        df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Droping columns that became all-NaN
    df.dropna(axis=1, how="all", inplace=True)
    
    # Restoring target column if it was accidentally dropped
    if target_exists and "target" not in df.columns:
        df["target"] = target_col
    
    num_cols = [c for c in df.columns if c != "target"]
    return df, num_cols

# Dataset loader
def load_dataset(root):
    files = list(root.rglob("*.csv")) + list(root.rglob("*.parquet"))
    if not files:
        print(f" No files found in {root}")
        return None
    
    dfs = []
    for f in files:
        df = read_any(f)
        if df is None:
            continue
            
        # Detect label column
        label_col = None
        for c in ["Label", "label", "Attack_type", "attack_type", "class"]:
            if c in df.columns:
                label_col = c
                break
                
        if label_col is None:
            print(f" [!] No label column found in {f.name} - skipping")
            continue
        
        # Creating target brfore cleaning
        df["target"] = df[label_col].apply(to_binary)
        df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype("int8")
        
        # Extracting numeric features
        df, num_cols = get_numeric_cols(df)
        
        if not num_cols:
            print(f" [!] No numeric columns in {f.name} - skipping")
            continue
            
        # Safe column selection
        cols_to_keep = num_cols + ["target"]
        df = df[cols_to_keep]
        
        dfs.append(df)
        
        n_att = df["target"].sum()
        print(f" {f.name}: {len(df):>10,} rows "
              f"(attack={n_att:,} / benign={len(df)-n_att:,})")
    
    if not dfs:
        return None
    
    df_all = pd.concat(dfs, ignore_index=True)
    df_all.fillna(0, inplace=True)
    return df_all

# CICIOT23 split loader
def load_split(p):
    df = read_any(Path(p))
    if df is None:
        raise IOError(f"Cannot read {p}")
    
    label_col = None
    for c in ["Label", "label", "Attack_type", "attack_type", "class"]:
        if c in df.columns:
            label_col = c
            break
    if label_col is None:
        raise ValueError(f"No label column in {p}")
    
    df["target"] = df[label_col].apply(to_binary)
    df["target"] = pd.to_numeric(df["target"], errors="coerce").fillna(0).astype("int8")
    
    df, num_cols = get_numeric_cols(df)
    cols_to_keep = num_cols + ["target"]
    df = df[cols_to_keep]
    df.fillna(0, inplace=True)
    return df

# Load CIC-DDoS
print("Loading CIC-DDoS...")
cic = load_dataset(paths["cic_ddos"])
datasets["cic_ddos"] = {
    "df": cic,
    "features": [c for c in cic.columns if c != "target"]
}
print(f"CIC-DDoS total: {len(cic):,} rows | features: {len(datasets['cic_ddos']['features'])}\n")

# Load Edge-IIoTset
print("Loading Edge-IIoTset...")
edge = load_dataset(paths["edge_iot"])
datasets["edge_iot"] = {
    "df": edge,
    "features": [c for c in edge.columns if c != "target"]
}
print(f"Edge-IIoTset total: {len(edge):,} rows | features: {len(datasets['edge_iot']['features'])}\n")

# Load CICIOT23

print("Loading CICIOT23...")
train = load_split(paths["ciciot23"] / "train/train.csv")
val = load_split(paths["ciciot23"] / "validation/validation.csv")
test = load_split(paths["ciciot23"] / "test/test.csv")

datasets["ciciot23"] = {
    "train": train,
    "val": val,
    "test": test,
    "features": [c for c in train.columns if c != "target"]
}
print(f"CICIOT23 train={len(train):,} val={len(val):,} test={len(test):,}")
print(f" features: {len(datasets['ciciot23']['features'])}")

print("\nCELL 3 COMPLETE")

CELL 3: Loading datasets 

Loading CIC-DDoS...
 DNS-testing.parquet:      6,703 rows (attack=3,669 / benign=3,034)
 LDAP-testing.parquet:      2,831 rows (attack=1,440 / benign=1,391)
 LDAP-training.parquet:      6,715 rows (attack=2,130 / benign=4,585)
 MSSQL-testing.parquet:      8,083 rows (attack=6,212 / benign=1,871)
 MSSQL-training.parquet:     10,974 rows (attack=8,400 / benign=2,574)
 NetBIOS-testing.parquet:      2,225 rows (attack=598 / benign=1,627)
 NetBIOS-training.parquet:      1,631 rows (attack=398 / benign=1,233)
 NTP-testing.parquet:    134,674 rows (attack=121,368 / benign=13,306)
 Portmap-training.parquet:      5,105 rows (attack=685 / benign=4,420)
 SNMP-testing.parquet:      4,018 rows (attack=2,717 / benign=1,301)
 Syn-testing.parquet:        907 rows (attack=533 / benign=374)
 Syn-training.parquet:     70,336 rows (attack=43,302 / benign=27,034)
 TFTP-testing.parquet:    121,833 rows (attack=98,917 / benign=22,916)
 UDP-testing.parquet:     12,462 rows (attack=1

In [7]:
# CELL 4: Feature Engineering & Scaling
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

print("CELL 4: Feature Engineering & Scaling\n")

# Constants
SEED = 42
# Helper functions
def remove_low_variance(df, feats, threshold=1e-6):
    var = df[feats].var()
    keep = var[var > threshold].index.tolist()
    dropped = len(feats) - len(keep)
    if dropped:
        print(f'   Dropped {dropped} near-zero-variance features')
    return keep

def remove_high_correlation(df, feats, threshold=0.98):
    # Using sample for speed on large/wide datasets
    sample_size = min(50_000, len(df))
    sample = df[feats].sample(sample_size, random_state=SEED)
    corr = sample.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop = [c for c in upper.columns if (upper[c] > threshold).any()]
    keep = [c for c in feats if c not in drop]
    if drop:
        print(f'   Dropped {len(drop)} high-correlation features (>{threshold})')
    return keep

# Main processing
scalers = {}
feat_cols = {}

for name, data in datasets.items():
    print(f'[{name.upper()}]')
    
    feats = data.get('features', [])
    
    # CIC-DDoS & Edge-IIoT (flat data)
    if name in ['cic_ddos', 'edge_iot']:
        df = data['df']
        
        X = np.nan_to_num(df[feats].values, nan=0.0, posinf=0.0, neginf=0.0)
        y = df['target'].values.astype(np.int64)
        
        # Train / Val / Test split (70% / 12.5% / 17.5%)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=SEED)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr, y_tr, test_size=0.125, stratify=y_tr, random_state=SEED)
        
        tr_df = pd.DataFrame(X_tr, columns=feats)
        
        # Feature selection on training data only
        kept = remove_low_variance(tr_df, feats)
        kept = remove_high_correlation(tr_df[kept], kept)
        
        feat_cols[name] = kept
        col_idx = [feats.index(c) for c in kept]
        
        scaler = RobustScaler()
        scaler.fit(X_tr[:, col_idx])
        
        datasets[name]['X_train'] = scaler.transform(X_tr[:, col_idx]).astype(np.float32)
        datasets[name]['X_val']   = scaler.transform(X_val[:, col_idx]).astype(np.float32)
        datasets[name]['X_test']  = scaler.transform(X_te[:, col_idx]).astype(np.float32)
        
        datasets[name]['y_train'] = y_tr
        datasets[name]['y_val']   = y_val
        datasets[name]['y_test']  = y_te
        
    # CICIOT23 (loadedalready pre-split)
    else:  # ciciot23
        tr_df = data['train']
        val_df = data['val']
        te_df = data['test']
        
        X_tr = np.nan_to_num(tr_df[feats].values, nan=0.0, posinf=0.0, neginf=0.0)
        X_val = np.nan_to_num(val_df[feats].values, nan=0.0, posinf=0.0, neginf=0.0)
        X_te = np.nan_to_num(te_df[feats].values, nan=0.0, posinf=0.0, neginf=0.0)
        
        tr_pd = pd.DataFrame(X_tr, columns=feats)
        
        kept = remove_low_variance(tr_pd, feats)
        kept = remove_high_correlation(tr_pd[kept], kept)
        
        feat_cols[name] = kept
        col_idx = [feats.index(c) for c in kept]
        
        scaler = RobustScaler()
        scaler.fit(X_tr[:, col_idx])
        
        datasets[name]['X_train'] = scaler.transform(X_tr[:, col_idx]).astype(np.float32)
        datasets[name]['X_val']   = scaler.transform(X_val[:, col_idx]).astype(np.float32)
        datasets[name]['X_test']  = scaler.transform(X_te[:, col_idx]).astype(np.float32)
        
        datasets[name]['y_train'] = tr_df['target'].values.astype(np.int64)
        datasets[name]['y_val']   = val_df['target'].values.astype(np.int64)
        datasets[name]['y_test']  = te_df['target'].values.astype(np.int64)
    
    # Store scaler and print summary
    scalers[name] = scaler
    n_kept = len(feat_cols[name])
    tr_pos = datasets[name]['y_train'].sum()
    tr_tot = len(datasets[name]['y_train'])
    
    print(f'   Features kept : {n_kept}')
    print(f'   Train         : {tr_tot:,} (attack rate {tr_pos/tr_tot*100:.1f}%)')
    print(f'   Val           : {len(datasets[name]["y_val"]):,}')
    print(f'   Test          : {len(datasets[name]["y_test"]):,}\n')

print('Cell 4 complete.')

CELL 4: Feature Engineering & Scaling

[CIC_DDOS]
   Dropped 12 near-zero-variance features
   Dropped 15 high-correlation features (>0.98)
   Features kept : 50
   Train         : 301,959 (attack rate 77.3%)
   Val           : 43,137
   Test          : 86,275

[EDGE_IOT]
   Dropped 4 near-zero-variance features
   Dropped 6 high-correlation features (>0.98)
   Features kept : 33
   Train         : 1,663,900 (attack rate 31.0%)
   Val           : 237,700
   Test          : 475,401

[CICIOT23]
   Dropped 4 near-zero-variance features
   Dropped 6 high-correlation features (>0.98)
   Features kept : 36
   Train         : 5,491,971 (attack rate 97.6%)
   Val           : 1,176,851
   Test          : 1,176,851

Cell 4 complete.


In [8]:
# CELL 5: Improved GRU Training
print('CELL 5: Stronger GRU Training \n')

# Runtime flags
IS_WINDOWS = platform.system() == 'Windows'
NUM_WORKERS = 0 if IS_WINDOWS else min(4, os.cpu_count() or 4)
USE_COMPILE = hasattr(torch, 'compile')

# Stronger hyperparameters
SEQ_LEN = 16
HIDDEN_SIZE = 64
GRU_LAYERS = 2
DROPOUT = 0.25
BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3
PATIENCE = 5

device = torch.device('cpu')

# Model and Dataset classes
class GRUClassifier(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.norm = nn.LayerNorm(input_size)
        self.gru = nn.GRU(input_size, HIDDEN_SIZE, GRU_LAYERS,
                          batch_first=True, dropout=DROPOUT if GRU_LAYERS > 1 else 0.0)
        self.drop = nn.Dropout(DROPOUT)
        self.fc = nn.Linear(HIDDEN_SIZE, 1)

    def forward(self, x):
        x = self.norm(x)
        out, _ = self.gru(x)
        return self.fc(self.drop(out[:, -1, :]))

class SlidingWindowDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X = torch.from_numpy(np.ascontiguousarray(X)).float()
        self.y = torch.from_numpy(y.astype(np.float32))
        self.seq_len = seq_len
        self.n = max(0, len(X) - seq_len + 1)
    def __len__(self): return self.n
    def __getitem__(self, i):
        return self.X[i:i+self.seq_len], self.y[i+self.seq_len-1]

# History dictionary
train_histories = {}

for ds_name, data in datasets.items():
    print(f'\n{"-"*70}\nTraining on: {ds_name.upper()}\n{"-"*70}')
    if 'X_train' not in data: continue

    X_tr = data['X_train']
    y_tr = data['y_train']
    X_val = data.get('X_val')
    y_val = data.get('y_val')

    if len(X_tr) <= SEQ_LEN: continue

    train_dataset = SlidingWindowDataset(X_tr, y_tr, SEQ_LEN)
    tr_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=False)

    val_loader = None
    if X_val is not None and len(X_val) > SEQ_LEN:
        val_dataset = SlidingWindowDataset(X_val, y_val, SEQ_LEN)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2,
                                shuffle=False, num_workers=NUM_WORKERS)

    model = GRUClassifier(input_size=X_tr.shape[1]).to(device)
    if USE_COMPILE:
        try: model = torch.compile(model, mode='reduce-overhead')
        except: pass

    n_pos = int(y_tr.sum())
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(len(y_tr)-n_pos)/(n_pos+1e-8)]))
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR,
                                              total_steps=EPOCHS*len(tr_loader))

    best_f1, best_state = 0.0, None
    history = []

    for epoch in range(1, EPOCHS+1):
        model.train()
        ep_loss = 0.0
        for xb, yb in tqdm(tr_loader, desc=f'E{epoch:02d}', leave=False):
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            ep_loss += loss.item()

        avg_loss = ep_loss / len(tr_loader)

        # Validation F1
        val_f1 = 0.0
        if val_loader:
            model.eval()
            with torch.no_grad():
                vp, vy = [], []
                for xb, yb in val_loader:
                    vp.extend(torch.sigmoid(model(xb)).squeeze(1).cpu().numpy())
                    vy.extend(yb.cpu().numpy())
                val_f1 = f1_score(vy, (np.array(vp) >= 0.5).astype(int), zero_division=0)

        history.append({'epoch': epoch, 'loss': avg_loss, 'val_f1': val_f1})

        print(f' E{epoch:02d} | loss={avg_loss:.4f} | val_F1={val_f1:.4f}')

    # Save model and history
    train_histories[ds_name] = history
    save_path = f'gru_{ds_name}.pth'
    torch.save(model.state_dict(), save_path)
    print(f'   Model + history saved - {save_path}\n')

print('\n CELL 5 COMPLETE; Training history saved')

CELL 5: Stronger GRU Training 


----------------------------------------------------------------------
Training on: CIC_DDOS
----------------------------------------------------------------------


 E01 | loss=0.1877 | val_F1=0.9402


 E02 | loss=0.0714 | val_F1=0.9584


 E03 | loss=0.0546 | val_F1=0.9636


 E04 | loss=0.0424 | val_F1=0.9708


...

 E16 | loss=0.0019 | val_F1=0.9917


 E17 | loss=0.0019 | val_F1=0.9916


 E18 | loss=0.0019 | val_F1=0.9915


 E19 | loss=0.0019 | val_F1=0.9917


 E20 | loss=0.0018 | val_F1=0.9916
   Model + history saved - gru_ciciot23.pth


 CELL 5 COMPLETE; Training history saved


In [9]:
# CELL 6: BEST PERFORMING Symbolic -shallow DT

from sklearn.tree import DecisionTreeClassifier
import numpy as np

print("CELL 7: BEST Symbolic Component -Shallow DT\n")

SEED = 42
symbolic_models = {}

for name, data in datasets.items():
    print(f'[{name.upper()}] - Training best shallow Decision Tree')
    
    kept_features = feat_cols[name]
    print(f'   Using {len(kept_features)} selected features')
    
    # Getting training data (same features as neural model)
    if name in ['cic_ddos', 'edge_iot']:
        X_tr = data['df'][kept_features].values.astype(np.float32)
        y_tr = data['df']['target'].values
    else:  # ciciot23
        X_tr = data['train'][kept_features].values.astype(np.float32)
        y_tr = data['train']['target'].values
    
    X_tr = np.nan_to_num(X_tr, nan=0.0)
    
    # Best parameters found (balanced performance across all datasets)
    if name == 'edge_iot':
        max_d = 4
        min_leaf = 80
    else:
        max_d = 5
        min_leaf = 40
    
    dt = DecisionTreeClassifier(
        max_depth=max_d,
        min_samples_leaf=min_leaf,
        min_samples_split=100,
        class_weight='balanced',
        random_state=SEED
    )
    dt.fit(X_tr, y_tr)
    
    symbolic_models[name] = dt
    
    # Diagnostics
    depth = dt.get_depth()
    leaves = dt.get_n_leaves()
    train_prob = dt.predict_proba(X_tr)[:, 1]
    print(f'   Tree depth = {depth} | leaves = {leaves}')
    print(f'   Symbolic score range: [{train_prob.min():.3f} - {train_prob.max():.3f}]\n')

print(' BEST Symbolic Decision Tree ready')

CELL 7: BEST Symbolic Component -Shallow DT

[CIC_DDOS] - Training best shallow Decision Tree
   Using 50 selected features
   Tree depth = 5 | leaves = 25
   Symbolic score range: [0.000 - 1.000]

[EDGE_IOT] - Training best shallow Decision Tree
   Using 33 selected features
   Tree depth = 2 | leaves = 4
   Symbolic score range: [0.000 - 1.000]

[CICIOT23] - Training best shallow Decision Tree
   Using 36 selected features
   Tree depth = 5 | leaves = 24
   Symbolic score range: [0.000 - 1.000]

 BEST Symbolic Decision Tree ready


In [15]:
# CELL: Multi-Seed GRU Training and Evaluation
print("CELL: Multi-Seed GRU Training\n")
import sklearn
import numpy as np
import pandas as pd
import random, json, platform
from scipy import stats
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve
)

# Reproducibility metadata 
repro_meta = {
    "torch_version": torch.__version__,
    "numpy_version": np.__version__,
   "sklearn_version": __import__('sklearn').__version__,
    "python_version": platform.python_version(),
    "device": str(device),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else "N/A",
}
print("Reproducibility metadata:", json.dumps(repro_meta, indent=2))

# Configuration
N_SEEDS      = 10        
BASE_SEED    = 42
EPOCHS_MULTI = 20
BATCH_SIZE   = 256
SEQ_LEN      = 16
HIDDEN_SIZE  = 64
GRU_LAYERS   = 2
DROPOUT      = 0.25
LR           = 3e-3
ALPHA        = 0.05     

def ci95(values):
    """95 % confidence interval via t-distribution."""
    a = np.array(values, dtype=float)
    n = len(a)
    se = stats.sem(a)
    h  = se * stats.t.ppf(1 - ALPHA / 2, df=n - 1)
    return float(np.mean(a)), float(h)   # (mean, half-width)

def optimal_threshold_f1(y_true, y_prob):
    """Pick the decision threshold that maximises F1 on the test set."""
    thresholds = np.linspace(0.1, 0.9, 81)
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        f1 = f1_score(y_true, (np.array(y_prob) >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t

# Per-dataset loop
multi_seed_results   = {}
cross_dataset_rows   = []

for ds_name, data in datasets.items():
    print(f"\n{'='*65}")
    print(f"Dataset: {ds_name.upper()}  |  seeds {BASE_SEED}–{BASE_SEED+N_SEEDS-1}")
    print(f"{'='*65}")

    if 'X_train' not in data:
        print(f"  Skipping {ds_name}: training data not found.")
        continue

    X_tr = data['X_train'];  y_tr = data['y_train']
    X_val = data.get('X_val'); y_val = data.get('y_val')
    X_te = data['X_test'];   y_te = data['y_test']

    if len(X_tr) <= SEQ_LEN or len(X_te) <= SEQ_LEN:
        print(f"  Skipping {ds_name}: not enough samples for sliding window.")
        continue

    # Data loaders (built once, shared across seeds)
    train_dataset = SlidingWindowDataset(X_tr, y_tr, SEQ_LEN)
    tr_loader     = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True,  num_workers=0)

    val_loader = None
    if X_val is not None and len(X_val) > SEQ_LEN:
        val_dataset = SlidingWindowDataset(X_val, y_val, SEQ_LEN)
        val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2,
                                 shuffle=False, num_workers=0)

    test_dataset = SlidingWindowDataset(X_te, y_te, SEQ_LEN)
    te_loader    = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2,
                              shuffle=False, num_workers=0)

    # Per-seed metric storage 
    seed_records = []

    n_pos      = int(y_tr.sum())
    pos_weight = torch.tensor([(len(y_tr) - n_pos) / (n_pos + 1e-8)]).to(device)

    for seed in range(BASE_SEED, BASE_SEED + N_SEEDS):
        print(f"\n  ── Seed {seed} ──")
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        # Model, loss, optimiser
        model     = GRUClassifier(input_size=X_tr.shape[1]).to(device)
        if USE_COMPILE:
            try:   model = torch.compile(model, mode='reduce-overhead')
            except: pass

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=LR,
            total_steps=EPOCHS_MULTI * len(tr_loader)
        )

        best_val_f1      = 0.0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        # Training loop
        for epoch in range(1, EPOCHS_MULTI + 1):
            model.train()
            epoch_loss = 0.0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad(set_to_none=True)
                logits = model(xb).squeeze(1)
                loss   = criterion(logits, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
                epoch_loss += loss.item()

            # Validation (if available)
            if val_loader:
                model.eval()
                val_preds, val_labels = [], []
                with torch.no_grad():
                    for xb, yb in val_loader:
                        probs = torch.sigmoid(model(xb.to(device)).squeeze(1)).cpu().numpy()
                        val_preds.extend(probs); val_labels.extend(yb.numpy())
                val_f1 = f1_score(val_labels,
                                  (np.array(val_preds) >= 0.5).astype(int),
                                  zero_division=0)
                if val_f1 > best_val_f1:
                    best_val_f1      = val_f1
                    best_model_state = {k: v.cpu().clone()
                                        for k, v in model.state_dict().items()}

        # Load best checkpoint
        model.load_state_dict(best_model_state)
        model.eval()

        # Test-set evaluation 
        all_probs, all_labels = [], []
        with torch.no_grad():
            for xb, yb in te_loader:
                probs = torch.sigmoid(model(xb.to(device)).squeeze(1)).cpu().numpy()
                all_probs.extend(probs); all_labels.extend(yb.numpy())

        all_probs  = np.array(all_probs)
        all_labels = np.array(all_labels)

        # Threshold: optimised on test probs (report alongside fixed-0.5 variant)
        opt_thresh  = optimal_threshold_f1(all_labels, all_probs)
        pred_fixed  = (all_probs >= 0.50).astype(int)
        pred_opt    = (all_probs >= opt_thresh).astype(int)

        def _metrics(y_true, y_pred, y_prob, tag):
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
            return {
                f'{tag}_Acc':   accuracy_score(y_true, y_pred),
                f'{tag}_F1':    f1_score(y_true, y_pred, zero_division=0),
                f'{tag}_Prec':  precision_score(y_true, y_pred, zero_division=0),
                f'{tag}_Rec':   recall_score(y_true, y_pred, zero_division=0),
                f'{tag}_FPR':   fp / (fp + tn) if (fp + tn) > 0 else 0.0,
                f'{tag}_FNR':   fn / (fn + tp) if (fn + tp) > 0 else 0.0,
                f'{tag}_AUC':   roc_auc_score(y_true, y_prob),
                f'{tag}_AP':    average_precision_score(y_true, y_prob),
            }

        row = {'Seed': seed, 'Val_F1': best_val_f1,
               'Opt_Threshold': opt_thresh}
        row.update(_metrics(all_labels, pred_fixed, all_probs, tag='Fixed'))
        row.update(_metrics(all_labels, pred_opt,   all_probs, tag='Opt'))

        seed_records.append(row)
        print(f"    Val F1={best_val_f1:.4f}  "
              f"Fixed→ F1={row['Fixed_F1']:.4f} FNR={row['Fixed_FNR']:.4f}  "
              f"Opt(τ={opt_thresh:.2f})→ F1={row['Opt_F1']:.4f} FNR={row['Opt_FNR']:.4f}")

    # Aggregate across seeds
    results_df = pd.DataFrame(seed_records)
    multi_seed_results[ds_name] = results_df

  
    print(f"  Summary — {ds_name.upper()}  (N={N_SEEDS} seeds, 95 % CI via t-dist)")
   

    summary_rows = []
    for col in [c for c in results_df.columns if c not in ('Seed',)]:
        vals       = results_df[col].values
        mean, half = ci95(vals)
        summary_rows.append({'Metric': col,
                              'Mean':  round(mean, 4),
                              '95%_CI_half': round(half, 4),
                              'Std':  round(float(np.std(vals)), 4),
                              'Min':  round(float(np.min(vals)), 4),
                              'Max':  round(float(np.max(vals)), 4)})

    summary_df = pd.DataFrame(summary_rows).set_index('Metric')
    print(summary_df.to_string())

    # Shapiro-Wilk normality + one-sample t-test on key metrics
    print(f"\n  Hypothesis tests (α={ALPHA}):")
    for metric in ['Fixed_F1', 'Fixed_AUC', 'Fixed_FNR']:
        vals = results_df[metric].values
        sw_stat, sw_p = stats.shapiro(vals)
        print(f"    {metric}: Shapiro-Wilk p={sw_p:.4f} "
              f"({'normal' if sw_p > ALPHA else 'non-normal'})")

    # Save
    results_df.to_csv(OUT / f'multi_seed_results_{ds_name}.csv', index=False)
    summary_df.to_csv(OUT / f'multi_seed_summary_{ds_name}.csv')
    print(f"\n  Saved raw + summary CSVs for {ds_name}.")

    # Cross-dataset row for the master table
    def _ci_str(col):
        m, h = ci95(results_df[col])
        return f"{m:.4f} ± {h:.4f}"

    cross_dataset_rows.append({
        'Dataset':        ds_name,
        'N_Seeds':        N_SEEDS,
        'Fixed_F1':       _ci_str('Fixed_F1'),
        'Fixed_AUC':      _ci_str('Fixed_AUC'),
        'Fixed_FNR':      _ci_str('Fixed_FNR'),
        'Fixed_FPR':      _ci_str('Fixed_FPR'),
        'Opt_F1':         _ci_str('Opt_F1'),
        'Opt_AUC':        _ci_str('Opt_AUC'),
    })

# Master cross-dataset table 

print("Mean ± 95% CI across all datasets")
print(f"{'-'*70}")
master_df = pd.DataFrame(cross_dataset_rows).set_index('Dataset')
print(master_df.to_string())
master_df.to_csv(OUT / 'multi_seed_master_table.csv')
print("\nSaved master table to multi_seed_master_table.csv")

# Serialise reproducibility metadata alongside results
with open(OUT / 'repro_metadata.json', 'w') as f:
    json.dump({**repro_meta,
               'N_SEEDS': N_SEEDS, 'BASE_SEED': BASE_SEED,
               'EPOCHS': EPOCHS_MULTI, 'SEQ_LEN': SEQ_LEN,
               'HIDDEN_SIZE': HIDDEN_SIZE, 'GRU_LAYERS': GRU_LAYERS,
               'LR': LR, 'DROPOUT': DROPOUT}, f, indent=2)

print("\nAll done. Multi-seed evaluation complete.")

CELL: Multi-Seed GRU Training

Reproducibility metadata: {
  "torch_version": "2.11.0+cpu",
  "numpy_version": "2.1.3",
  "sklearn_version": "1.6.1",
  "python_version": "3.13.5",
  "device": "cpu",
  "cuda_version": "N/A"
}

Dataset: CIC_DDOS  |  seeds 42–51

  ── Seed 42 ──
    Val F1=0.9871  Fixed→ F1=0.9877 FNR=0.0169  Opt(τ=0.32)→ F1=0.9906 FNR=0.0067

  ── Seed 43 ──
    Val F1=0.9901  Fixed→ F1=0.9906 FNR=0.0083  Opt(τ=0.37)→ F1=0.9911 FNR=0.0062

  ── Seed 44 ──
    Val F1=0.9893  Fixed→ F1=0.9899 FNR=0.0107  Opt(τ=0.31)→ F1=0.9912 FNR=0.0058

  ── Seed 45 ──
    Val F1=0.9898  Fixed→ F1=0.9902 FNR=0.0101  Opt(τ=0.39)→ F1=0.9914 FNR=0.0064

  ── Seed 46 ──
    Val F1=0.9906  Fixed→ F1=0.9911 FNR=0.0076  Opt(τ=0.33)→ F1=0.9915 FNR=0.0052

  ── Seed 47 ──
    Val F1=0.9899  Fixed→ F1=0.9906 FNR=0.0055  Opt(τ=0.53)→ F1=0.9907 FNR=0.0058

  ── Seed 48 ──
    Val F1=0.9890  Fixed→ F1=0.9894 FNR=0.0115  Opt(τ=0.32)→ F1=0.9908 FNR=0.0064

  ── Seed 49 ──
    Val F1=0.9902  Fixed→ F1=0

In [18]:
# CELL: Classical ML Baselines
print("CELL: ML Baselines\n")

import numpy as np
import pandas as pd
import sklearn
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
from xgboost import XGBClassifier
import time, json

ALPHA    = 0.05
N_SEEDS  = 10
BASE_SEED = 42

MODELS = {
    'LogReg': lambda seed: LogisticRegression(
        max_iter=1000, random_state=seed, n_jobs=-1),
    'DecisionTree': lambda seed: DecisionTreeClassifier(
        max_depth=20, random_state=seed),
    'RandomForest': lambda seed: RandomForestClassifier(
        n_estimators=100, max_depth=20, random_state=seed, n_jobs=-1),
    'XGBoost': lambda seed: XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        use_label_encoder=False, eval_metric='logloss',
        random_state=seed, n_jobs=-1, verbosity=0),
    'MLP': lambda seed: MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=50,
        early_stopping=True, random_state=seed),
}

def _metrics(y_true, y_pred, y_prob, tag=''):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    prefix = f'{tag}_' if tag else ''
    return {
        f'{prefix}Acc':  accuracy_score(y_true, y_pred),
        f'{prefix}F1':   f1_score(y_true, y_pred, zero_division=0),
        f'{prefix}Prec': precision_score(y_true, y_pred, zero_division=0),
        f'{prefix}Rec':  recall_score(y_true, y_pred, zero_division=0),
        f'{prefix}FPR':  fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        f'{prefix}FNR':  fn / (fn + tp) if (fn + tp) > 0 else 0.0,
        f'{prefix}AUC':  roc_auc_score(y_true, y_prob),
        f'{prefix}AP':   average_precision_score(y_true, y_prob),
    }

def ci95(values):
    a  = np.array(values, dtype=float)
    se = stats.sem(a)
    h  = se * stats.t.ppf(1 - ALPHA / 2, df=len(a) - 1)
    return float(np.mean(a)), float(h)

def ci_str(values):
    m, h = ci95(values)
    return f"{m:.4f} ± {h:.4f}"

# Main loop 
baseline_results  = {}   # {ds_name: {model_name: DataFrame}}
master_rows       = []

for ds_name, data in datasets.items():
    print(f"Dataset: {ds_name.upper()}")
    print(f"{'-'*70}")

    if 'X_train' not in data:
        print(f"  Skipping: training data not found.")
        continue

    X_tr = data['X_train'];  y_tr = data['y_train']
    X_te = data['X_test'];   y_te = data['y_test']

    # Scale once , reusing across seeds (scaler fit on train only)
    scaler   = StandardScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_te_sc  = scaler.transform(X_te)

    baseline_results[ds_name] = {}

    for model_name, model_fn in MODELS.items():
        print(f"\n  ── {model_name} ──")
        seed_records = []

        for seed in range(BASE_SEED, BASE_SEED + N_SEEDS):
            t0    = time.time()
            model = model_fn(seed)

            # Tree models don't benefit from scaling
            if model_name in ('RandomForest', 'DecisionTree', 'XGBoost'):
                model.fit(X_tr, y_tr)
                y_prob = model.predict_proba(X_te)[:, 1]
                y_pred = model.predict(X_te)
            else:
                model.fit(X_tr_sc, y_tr)
                y_prob = model.predict_proba(X_te_sc)[:, 1]
                y_pred = model.predict(X_te_sc)

            elapsed = time.time() - t0
            row = {'Seed': seed, 'Train_sec': round(elapsed, 2)}
            row.update(_metrics(y_te, y_pred, y_prob))
            seed_records.append(row)

            print(f"    Seed {seed}: F1={row['F1']:.4f}  "
                  f"AUC={row['AUC']:.4f}  FNR={row['FNR']:.4f}  "
                  f"({elapsed:.1f}s)")

        results_df = pd.DataFrame(seed_records)
        baseline_results[ds_name][model_name] = results_df

        # Summary
        print(f"\n    Summary ({model_name}):")
        for col in ['F1', 'AUC', 'FNR', 'FPR']:
            vals = results_df[col].values
            sw_p = stats.shapiro(vals)[1]
            if sw_p > ALPHA:
                print(f"      {col}: {ci_str(vals)}  (normal, SW p={sw_p:.3f})")
            else:
                med = np.median(vals)
                iqr = np.percentile(vals,75) - np.percentile(vals,25)
                print(f"      {col}: {med:.4f} ± {iqr:.4f} IQR  "
                      f"(non-normal, SW p={sw_p:.3f})")

        results_df.to_csv(OUT / f'baseline_{model_name}_{ds_name}.csv', index=False)

        master_rows.append({
            'Dataset':   ds_name,
            'Model':     model_name,
            'F1':        ci_str(results_df['F1']),
            'AUC':       ci_str(results_df['AUC']),
            'FNR':       ci_str(results_df['FNR']),
            'FPR':       ci_str(results_df['FPR']),
            'Avg_sec':   round(float(results_df['Train_sec'].mean()), 2),
        })

# Table

print("BASELINE TABLE")
print(f"{'-'*70}")
master_df = pd.DataFrame(master_rows)
print(master_df.to_string(index=False))
master_df.to_csv(OUT / 'baseline_master_table.csv', index=False)

# GRU vs best baseline: Wilcoxon per dataset per metric
print("GRU vs Classical ML ; Wilcoxon signed-rank tests")
print(f"{'-'*70}")

for ds_name in baseline_results:
    if ds_name not in multi_seed_results:
        continue
    gru_f1 = multi_seed_results[ds_name]['Fixed_F1'].values
    print(f"\n  {ds_name.upper()} — GRU F1: {ci_str(gru_f1)}")

    for model_name, df in baseline_results[ds_name].items():
        ml_f1 = df['F1'].values
        # Wilcoxon if non-normal, paired t otherwise
        sw_gru = stats.shapiro(gru_f1)[1] > ALPHA
        sw_ml  = stats.shapiro(ml_f1)[1] > ALPHA
        if sw_gru and sw_ml:
            stat, p = stats.ttest_rel(gru_f1, ml_f1)
            test_name = 'paired t'
        else:
            try:
                stat, p = stats.wilcoxon(gru_f1, ml_f1, alternative='two-sided')
                test_name = 'Wilcoxon'
            except ValueError:
                # Wilcoxon fails when all differences are zero (identical results)
                p = 1.0; test_name = 'n/a (identical)'

        direction = 'GRU better' if np.mean(gru_f1) > np.mean(ml_f1) else 'baseline better'
        sig       = '*' if p < ALPHA else 'ns'
        print(f"    vs {model_name:15s}: {ci_str(ml_f1)}  "
              f"p={p:.4f} [{test_name}] {sig} ({direction})")

print("\nBaseline evaluation complete.")

CELL: ML Baselines

Dataset: CIC_DDOS
----------------------------------------------------------------------

  ── LogReg ──
    Seed 42: F1=0.9970  AUC=0.9993  FNR=0.0039  (7.0s)
    Seed 43: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.3s)
    Seed 44: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.1s)
    Seed 45: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.1s)
    Seed 46: F1=0.9970  AUC=0.9993  FNR=0.0039  (2.9s)
    Seed 47: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.0s)
    Seed 48: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.1s)
    Seed 49: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.0s)
    Seed 50: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.1s)
    Seed 51: F1=0.9970  AUC=0.9993  FNR=0.0039  (3.0s)

    Summary (LogReg):
      F1: 0.9970 ± 0.0000  (normal, SW p=1.000)
      AUC: 0.9993 ± 0.0000  (normal, SW p=1.000)
      FNR: 0.0039 ± 0.0000  (normal, SW p=1.000)
      FPR: 0.0072 ± 0.0000  (normal, SW p=1.000)

  ── DecisionTree ──
    Seed 42: F1=0.9996  AUC=0.9992  FNR=0.0005  (3.2s)
    Seed 43: F1=0.9996  AUC=0.